# Fine-tuning Granite 3.3-8b-instruct Model with LoRA

This notebook fine-tunes the IBM Granite 3.3-8b-instruct model using LoRA (Low-Rank Adaptation) on NL2SQL datasets.

## Features included:
- Supports multiple training datasets (train.json and train_creditcard.json)
- Uses LoRA for efficient fine-tuning
- Configurable training parameters
- Error handling and validation

## 1. Install Dependencies

In [29]:
# Install required packages
%pip install -q llamafactory 2>/dev/null
%pip install -q --upgrade numpy 2>/dev/null
%pip install -q --upgrade pandas 2>/dev/null

print("✓ Dependencies installed successfully")

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
✓ Dependencies installed successfully


## 2. Check GPU Environment

In [1]:
import torch
import platform

# Check for GPU availability (CUDA for NVIDIA, MPS for Apple Silicon)
if torch.cuda.is_available():
    print(f"✓ CUDA GPU available: {torch.cuda.get_device_name(0)}")
    print(f"  CUDA version: {torch.version.cuda}")
    print(f"  GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    device = "cuda"
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    print(f"✓ Apple Silicon GPU (MPS) available")
    print(f"  Device: {platform.processor()}")
    print(f"  System: {platform.system()} {platform.release()}")
    device = "mps"
else:
    print("⚠ WARNING: No GPU found. Training will be very slow on CPU.")
    print("  Consider using a GPU-enabled environment for better performance.")
    device = "cpu"

print(f"\nUsing device: {device}")

✓ Apple Silicon GPU (MPS) available
  Device: arm
  System: Darwin 24.6.0

Using device: mps


## 3. Configuration

In [2]:
import os

# Configuration
CONFIG = {
    # Data paths
    "train_files": [
        "../data/train_aml.json",
        "../data/train_creditcard.json"
    ],
    "output_file": "../data/output.json",
    "dataset_info_file": "../data/dataset_info.json",
    
    # Model configuration
    "model_path": "ibm-granite/granite-3.3-8b-instruct",  # Granite 3.3 8B Instruct
    "template": "granite3",  # Using granite3 template
    
    # Training configuration
    "output_dir": "granite33_8b_lora",
    "num_train_epochs": 3.0,
    "per_device_train_batch_size": 2,  # Reduced for 8B model
    "gradient_accumulation_steps": 4,  # Increased to compensate
    "learning_rate": 1e-4,
    "max_samples": 500,
    
    # LoRA configuration
    "lora_target": "all",
    "loraplus_lr_ratio": 16.0,
}

print("✓ Configuration loaded")
print(f"  Training files: {len(CONFIG['train_files'])}")
print(f"  Model: {CONFIG['model_path']}")
print(f"  Output directory: {CONFIG['output_dir']}")

✓ Configuration loaded
  Training files: 2
  Model: ibm-granite/granite-3.3-8b-instruct
  Output directory: granite33_8b_lora


## 4. Load and Process Training Data

In [3]:
import pandas as pd
import json
from pathlib import Path

def load_and_format_data(file_paths):
    """
    Load multiple JSON files and format them to Alpaca format.
    
    Args:
        file_paths: List of paths to JSON training files
        
    Returns:
        List of formatted training examples
    """
    all_data = []
    
    for file_path in file_paths:
        if not Path(file_path).exists():
            print(f"⚠ Warning: File not found: {file_path}")
            continue
            
        try:
            # Read JSON file
            df = pd.read_json(file_path)
            
            # Format to Alpaca format (handle both 'SQL' and 'sql' columns)
            sql_col = 'SQL' if 'SQL' in df.columns else 'sql'
            formatted = [
                {
                    "instruction": row["question"],
                    "input": row.get("evidence", ""),  # Use evidence as additional context
                    "output": row[sql_col]
                }
                for _, row in df.iterrows()
            ]
            
            all_data.extend(formatted)
            print(f"✓ Loaded {len(formatted)} examples from {Path(file_path).name}")
            
        except Exception as e:
            print(f"✗ Error loading {file_path}: {str(e)}")
            continue
    
    return all_data

# Load and format all training data
formatted_data = load_and_format_data(CONFIG["train_files"])

if not formatted_data:
    raise ValueError("No training data loaded. Please check your file paths.")

print(f"\n✓ Total training examples: {len(formatted_data)}")
print("\nExample training instance:")
print(json.dumps(formatted_data[0], indent=2))

✓ Loaded 5 examples from train_aml.json
✓ Loaded 9 examples from train_creditcard.json

✓ Total training examples: 14

Example training instance:
{
  "instruction": "Give me the top 5 most similar accounts to the known suspicious accounts 00A3AB920 and 00BA278F0",
  "input": "Use AI_SEMANTIC_CLUSTER to treat the known suspicious accounts (00A3AB920 and 00BA278F0) as seed examples and then ranks other FROM_ACCOUNT values by how strongly they belong to the same behavioral cluster.",
  "output": "SELECT DISTINCT A.FROM_BANK, A.FROM_ACCOUNT, A.FROM_ACCOUNT_KYC, AI_SEMANTIC_CLUSTER( FROM_ACCOUNT, '00A3AB920', '00BA278F0') AS SIMILARITY FROM {schema}.{table} A ORDER BY SIMILARITY DESC FETCH FIRST 5 ROWS ONLY;"
}


## 5. Save Formatted Dataset

In [4]:
from pathlib import Path
import json

# Create output directory if it doesn't exist
output_path = Path(CONFIG["output_file"])
output_path.parent.mkdir(parents=True, exist_ok=True)

# Save formatted dataset
with open(CONFIG["output_file"], "w", encoding="utf-8") as f:
    json.dump(formatted_data, f, indent=2, ensure_ascii=False)

print(f"Formatted dataset saved to: {CONFIG['output_file']}")

# Create dataset_info.json for LLaMA Factory
dataset_info = {
    "nl2sql_combined": {
        "file_name": "output.json",
    }
}

dataset_info_path = Path(CONFIG["dataset_info_file"])
dataset_info_path.parent.mkdir(parents=True, exist_ok=True)

with open(CONFIG["dataset_info_file"], "w", encoding="utf-8") as f:
    json.dump(dataset_info, f, indent=2, ensure_ascii=False)

print(f"Dataset info saved to: {CONFIG['dataset_info_file']}")

Formatted dataset saved to: ../data/output.json
Dataset info saved to: ../data/dataset_info.json


## 6. Configure Training Parameters

In [5]:
import yaml

# Training arguments for LLaMA Factory
training_args = {
    "stage": "sft",  # Supervised fine-tuning
    "do_train": True,
    "model_name_or_path": CONFIG["model_path"],
    "dataset": "nl2sql_combined",
    "template": CONFIG["template"],
    "finetuning_type": "lora",
    "lora_target": CONFIG["lora_target"],
    "loraplus_lr_ratio": CONFIG["loraplus_lr_ratio"],
    "output_dir": CONFIG["output_dir"],
    "per_device_train_batch_size": CONFIG["per_device_train_batch_size"],
    "gradient_accumulation_steps": CONFIG["gradient_accumulation_steps"],
    "learning_rate": CONFIG["learning_rate"],
    "num_train_epochs": CONFIG["num_train_epochs"],
    "max_samples": CONFIG["max_samples"],
    "fp16": torch.cuda.is_available(),  # Use fp16 for CUDA
    "bf16": hasattr(torch.backends, 'mps') and torch.backends.mps.is_available(),  # Use bf16 for Apple Silicon
    "logging_steps": 10,
    "save_steps": 100,
    "report_to": "none",
    "overwrite_output_dir": True,
}

# Save training configuration
config_file = "train_granite33_8b_lora.yaml"
with open(config_file, "w", encoding="utf-8") as f:
    yaml.dump(training_args, f, indent=2)

print(f"✓ Training configuration saved to: {config_file}")
print("\nTraining parameters:")
for key, value in training_args.items():
    print(f"  {key}: {value}")

✓ Training configuration saved to: train_granite33_8b_lora.yaml

Training parameters:
  stage: sft
  do_train: True
  model_name_or_path: ibm-granite/granite-3.3-8b-instruct
  dataset: nl2sql_combined
  template: granite3
  finetuning_type: lora
  lora_target: all
  loraplus_lr_ratio: 16.0
  output_dir: granite33_8b_lora
  per_device_train_batch_size: 2
  gradient_accumulation_steps: 4
  learning_rate: 0.0001
  num_train_epochs: 3.0
  max_samples: 500
  fp16: False
  bf16: True
  logging_steps: 10
  save_steps: 100
  report_to: none
  overwrite_output_dir: True


## 7. Validate Configuration

In [6]:
# Validation checks
print("Running pre-training validation...\n")

checks_passed = True

# Check if model path exists (if it's a local path)
if Path(CONFIG["model_path"]).exists():
    print("Local model path exists")
else:
    print(f"INFO: Model path '{CONFIG['model_path']}' will be downloaded from HuggingFace")

# Check if output directory is writable
try:
    Path(CONFIG["output_dir"]).mkdir(parents=True, exist_ok=True)
    print("Output directory is writable")
except Exception as e:
    print(f"WARN: Cannot create output directory: {e}")
    checks_passed = False

# Check training data
if len(formatted_data) > 0:
    print(f"Training data loaded: {len(formatted_data)} examples")
else:
    print("ERROR: No training data available")
    checks_passed = False

# Check GPU for fp16 training
if training_args["fp16"] and not torch.cuda.is_available():
    print("WARN:: fp16 enabled but no GPU available")

if checks_passed:
    print("\n All validation checks passed. Ready to start training!")
else:
    print("\n Some validation checks failed. Please fix the issues before training.")

Running pre-training validation...

INFO: Model path 'ibm-granite/granite-3.3-8b-instruct' will be downloaded from HuggingFace
Output directory is writable
Training data loaded: 14 examples

 All validation checks passed. Ready to start training!


## 8. Start Training

**Note:** Training can take several hours depending on your hardware and dataset size.

In [7]:
# Start training with LLaMA Factory
print("Starting training...\n")
print("This may take a while. Monitor the progress below.\n")

!llamafactory-cli train train_granite33_8b_lora.yaml

Starting training...

This may take a while. Monitor the progress below.

[INFO|2026-02-10 21:57:20] llamafactory.hparams.parser:406 >> Process rank: 0, world size: 1, device: mps, distributed training: False, compute dtype: torch.bfloat16
[INFO|tokenization_utils_base.py:2060] 2026-02-10 21:57:21,023 >> loading file vocab.json from cache at /Users/hroy/.cache/huggingface/hub/models--ibm-granite--granite-3.3-8b-instruct/snapshots/51dd4bc2ade4059a6bd87649d68aa11e4fb2529b/vocab.json
[INFO|tokenization_utils_base.py:2060] 2026-02-10 21:57:21,023 >> loading file merges.txt from cache at /Users/hroy/.cache/huggingface/hub/models--ibm-granite--granite-3.3-8b-instruct/snapshots/51dd4bc2ade4059a6bd87649d68aa11e4fb2529b/merges.txt
[INFO|tokenization_utils_base.py:2060] 2026-02-10 21:57:21,023 >> loading file tokenizer.json from cache at /Users/hroy/.cache/huggingface/hub/models--ibm-granite--granite-3.3-8b-instruct/snapshots/51dd4bc2ade4059a6bd87649d68aa11e4fb2529b/tokenizer.json
[INFO|tokeniza

## 9. Post-Training Summary

In [8]:
import os
from pathlib import Path

output_dir = Path(CONFIG["output_dir"])

if output_dir.exists():
    print("✓ Training completed successfully!\n")
    print(f"Model artifacts saved to: {output_dir}\n")
    
    # List saved files
    print("Saved files:")
    for item in sorted(output_dir.iterdir()):
        if item.is_file():
            size = item.stat().st_size / (1024 * 1024)  # Convert to MB
            print(f"  - {item.name} ({size:.2f} MB)")
    
    print("\nNext steps:")
    print("1. Test the fine-tuned model using inference_trained_model.ipynb")
    print("2. Evaluate model performance on test data")
    print("3. Deploy the model for production use")
else:
    print("⚠ Training output directory not found. Training may have failed.")
    print("Please check the training logs above for errors.")

✓ Training completed successfully!

Model artifacts saved to: granite33_8b_lora

Saved files:
  - README.md (0.00 MB)
  - adapter_config.json (0.00 MB)
  - adapter_model.safetensors (94.45 MB)
  - added_tokens.json (0.00 MB)
  - all_results.json (0.00 MB)
  - merges.txt (0.42 MB)
  - special_tokens_map.json (0.00 MB)
  - tokenizer.json (3.32 MB)
  - tokenizer_config.json (0.01 MB)
  - train_results.json (0.00 MB)
  - trainer_log.jsonl (0.00 MB)
  - trainer_state.json (0.00 MB)
  - training_args.bin (0.01 MB)
  - vocab.json (0.74 MB)

Next steps:
1. Test the fine-tuned model using inference_trained_model.ipynb
2. Evaluate model performance on test data
3. Deploy the model for production use
